# SD 1.5 (.safetensors) → Qualcomm QNN `qnn2.39_min` — Colab

Bir **safetensors indirme linki** girin → dönüştürün → **Hugging Face reponuza** yükleyin.
Çıktı: `<isim>_qnn2.39_min.zip` (Ruya / Local Dream ile içe aktarılır).

### Önce oku (önemli):
1. **Runtime → Change runtime type → High-RAM** seçin. Ücretsiz katman (12 GB) 512px'te OOM olabilir; **Colab Pro / High-RAM (25–51 GB)** önerilir.
2. **QNN/QAIRT SDK otomatik indirilir** — `matrixportalx/qairt-sdk` **v2.39.0.250926** release'inden. Public olduğu için token gerekmez.
3. QAIRT araçları **Python 3.10** ister (Colab 3.12); 4. adım izole bir 3.10 ortamı kurar (otomatik).
4. HF token'ınızı Colab **Secrets** (🔑 sol menü) içine `HF_TOKEN` adıyla ekleyin (write izinli).
5. civitai linki token istiyorsa Secrets'a `CIVITAI_TOKEN` ekleyin.

Hücreleri **sırayla** çalıştırın.

## 1) Ayarlar (buradan doldurun)

In [ ]:
#@title Dönüşüm ayarları { display-mode: "form" }
SAFETENSORS_URL = ""  #@param {type:"string"}
MODEL_NAME = "MyModel"  #@param {type:"string"}
TIER = "min"  #@param ["min", "mid", "high"]
RESOLUTIONS = "512x512"  #@param ["512x512", "512x512,512x768,768x512"] {allow-input: true}

#@markdown **Gelişmiş / deneme** (normalde boş bırakın)
#@markdown - `DSP_ARCH`: hedef HTP mimarisi. Boş = tier varsayılanı (min→v69).
#@markdown   Cihazda yüklenmiyorsa **v68** deneyin (çalışan `_min` modeller muhtemelen v68).
#@markdown - `REBUILD_BIN`: kuantizasyonu TEKRARLAMADAN sadece .bin'leri yeniden üretir (~3 dk).
DSP_ARCH = ""  #@param ["", "v68", "v69", "v73", "v75"]
REBUILD_BIN = False  #@param {type:"boolean"}
#@markdown - `UNET_MODE`: UNet kuantizasyonu. **`a16w8_restrict` bırakın.**
#@markdown   Motor `sample`/`text_embedding`'e `uint16`, `timestamp`'e `int32`
#@markdown   yazar; bu yüzden **`a8w8` derlenir ama cihazda YÜKLENMEZ**
#@markdown   ("Motor süreci kapandı / Could not free context"). `a8w8` yalnızca
#@markdown   boru hattını test etmek için.
UNET_MODE = "a16w8_restrict"  #@param ["a16w8_restrict", "a16w8", "a8w8"]
#@markdown - `QUANT_EXTRA`: quantizer'a ek bayrak (kod değiştirmeden deneme).
#@markdown   Örn: `--target_backend HTP` veya `--act_quantizer_calibration mse`
QUANT_EXTRA = ""  #@param {type:"string"}

#@markdown **Hugging Face yükleme modu**
#@markdown - *Ayrı repo*: her model kendi reposuna → `<kullanıcı>/MODEL_NAME`
#@markdown - *Koleksiyon*: hepsi tek repoda, her model alt klasörde → `<kullanıcı>/COLLECTION/MODEL_NAME/`
UPLOAD_MODE = "Ayri repo (model adiyla)"  #@param ["Ayri repo (model adiyla)", "Koleksiyon (tek repo)"]
COLLECTION_REPO = "sd_qnn"  #@param {type:"string"}
HF_REPO = ""  #@param {type:"string"}
HF_PRIVATE = False  #@param {type:"boolean"}
#@markdown (`HF_REPO` doldurulursa iki modu da geçersiz kılar; doğrudan o repoya yükler.)

#@markdown **QAIRT SDK release** (varsayılan sizin public release'iniz)
QAIRT_REPO = "matrixportalx/qairt-sdk"  #@param {type:"string"}
QAIRT_TAG = "v2.39.0.250926"  #@param {type:"string"}
QNN_VERSION = "2.39"  #@param {type:"string"}

import os
os.environ["QNN_VERSION"] = QNN_VERSION
os.environ["DSP_ARCH"] = DSP_ARCH            # bos = tier varsayilani
os.environ["REBUILD_BIN"] = "1" if REBUILD_BIN else "0"
os.environ["UNET_MODE"] = UNET_MODE
os.environ["QUANT_EXTRA"] = QUANT_EXTRA
assert SAFETENSORS_URL, "SAFETENSORS_URL boş olamaz"
print("Ayarlar tamam:", MODEL_NAME, TIER, RESOLUTIONS, "| qnn", QNN_VERSION)
print("DSP_ARCH =", DSP_ARCH or "(tier varsayilani)", "| REBUILD_BIN =", REBUILD_BIN)
print("UNET_MODE =", UNET_MODE)

## 2) Depoyu çek + Python bağımlılıkları + MNN

In [ ]:
%cd /content
BR = "claude/qnn-model-conversion-snapdragon7-rsk8og"
URL = "https://github.com/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model.git"
# Klasor varsa en guncel kodu cek; yoksa klonla (re-run'da guncel requirements alinir)
!if [ -d sd-qnn ]; then cd sd-qnn && git fetch -q origin $BR && git reset -q --hard origin/$BR && cd ..; else git clone -q --branch $BR $URL sd-qnn; fi
%cd /content/sd-qnn
!pip -q install -r requirements.txt
# MNN converter (text_encoder + vae -> .mnn) + onnx export/sadelestirme araclari
!pip -q install MNN onnxscript onnxslim
import os
os.environ["MNNCONVERT"] = "mnnconvert"
print("OK")

## 2b) (Teşhis) Çalışan resmi bir `_min` modelin içindeki dosya yapısını göster

Bu hücre, referans bir modeli indirip **içindeki dosya listesini** yazar. Çıktıyı paylaşırsan paketleme (`05_package.py`) birebir buna göre düzeltilir. `HF_TOKEN` gerekmez (repo public ise).

In [ ]:
#@title Referans modelin dosya yapısını göster (teşhis)
REF_REPO = "Mr-J-369/AbyssOrangeMix3-SD1.5-qnn2.28"  #@param {type:"string"}
import os, zipfile
from huggingface_hub import HfApi, hf_hub_download
try:
    from google.colab import userdata
    t = userdata.get("HF_TOKEN")
    if t: os.environ["HF_TOKEN"] = t
except Exception:
    pass
tok = os.environ.get("HF_TOKEN")

api = HfApi(token=tok)
files = api.list_repo_files(REF_REPO)
print("REPO DOSYALARI:", files)
zips = [f for f in files if f.endswith(".zip")]
if not zips:
    print("\n[!] Repo'da zip yok. Dosyalar zaten acik olabilir (yukaridaki liste).")
else:
    target = next((z for z in zips if "min" in z.lower()), zips[0])
    print("\nIndiriliyor (icini gormek icin):", target)
    p = hf_hub_download(REF_REPO, target, token=tok)
    with zipfile.ZipFile(p) as zf:
        print(f"\n=== {target} ICERIGI ===")
        for n in zf.namelist():
            print(f"  {n}   ({zf.getinfo(n).file_size} bytes)")
print("\n>>> Bu ciktinin TAMAMINI paylas; paketleme buna gore duzeltilecek.")

## 3) (İsteğe bağlı) Swap ekle — düşük RAM'de OOM'u azaltır

In [ ]:
#@title 16 GB swap oluştur (ücretsiz katmanda önerilir)
!fallocate -l 16G /content/swapfile 2>/dev/null || dd if=/dev/zero of=/content/swapfile bs=1M count=16384
!chmod 600 /content/swapfile && mkswap /content/swapfile && swapon /content/swapfile
!free -h

## 4) QAIRT 2.39 SDK'yı indir + Python 3.10 ortamını kur (otomatik)

SDK release'ten iner; QAIRT araçları için izole bir Python 3.10 venv + libc++ kurulur. Release private ise Colab Secrets'a `GH_TOKEN` ekleyin (public ise gerekmez).

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ("GH_TOKEN", "GITHUB_TOKEN"):
        try:
            v = userdata.get(k)
            if v: os.environ["GH_TOKEN"] = v
        except Exception:
            pass
except Exception:
    pass

!python scripts/setup_qnn_sdk.py --repo "$QAIRT_REPO" --tag "$QAIRT_TAG" --dest /content/qairt | tee /content/sdk_setup.log

root = None
for line in open("/content/sdk_setup.log"):
    if line.startswith("QNN_SDK_ROOT="):
        root = line.strip().split("=", 1)[1]
assert root, "QNN_SDK_ROOT bulunamadı — 4. adım loguna bakın."
os.environ["QNN_SDK_ROOT"] = root
print("QNN_SDK_ROOT =", root)

# QAIRT python konvertorleri Python 3.10 + libc++ ister -> izole venv kur
!chmod +x scripts/*.sh
!bash scripts/setup_qnn_python.sh
if os.path.exists("/content/qnn_py.path"):
    os.environ["QNN_PYTHON"] = open("/content/qnn_py.path").read().strip()
    print("QNN_PYTHON =", os.environ["QNN_PYTHON"])

In [ ]:
#@title 4b) TEŞHİS: Çalışan referans modelin gerçek tensör tiplerini dök
#@markdown Uygulama UNet'e `uint16` latent + `int32` timestep yazıyor.
#@markdown Bu hücre çalışan bir referansın **gerçek** dtype'larını gösterir →
#@markdown ona göre üretiriz. (4. adımdan SONRA çalıştırın; SDK gerekli.)
REF_REPO = "Mr-J-369/AbyssOrangeMix3-SD1.5-qnn2.28"  #@param {type:"string"}
import os
try:
    from google.colab import userdata
    t = userdata.get("HF_TOKEN")
    if t: os.environ["HF_TOKEN"] = t
except Exception:
    pass
os.environ["REF_REPO"] = REF_REPO
!chmod +x scripts/*.sh
!bash scripts/inspect_reference.sh

## 4c) TEŞHİS (30 sn): QAIRT araçlarının TAM arayüzü
`unet.bin` üretimi v68'de 16-bit MatMul'u reddediyor. Bu hücre SDK'nın
gerçek seçeneklerini (mixed precision / backend-aware kuantizasyon) döker;
böylece 30 dk'lık deneme-yanılma yerine doğru bayrağı doğrudan kullanırız.
**Dönüşüm yapmaz, hızlıdır. Çıktının tamamını paylaşın.**

In [ ]:
#@title 4c) SDK arayüz dökümü (hızlı teşhis, ~30 sn)
!chmod +x scripts/*.sh
!bash scripts/dump_sdk_help.sh 2>&1 | tee /content/sdk_help.txt
import os
print('\n=== dosya:', os.path.getsize('/content/sdk_help.txt'), 'bayt ->',
      '/content/sdk_help.txt ===')

### 4d) Teşhis çıktısını Claude'a ulaştır — iki yol

**Yol 1 (kopyalama yok, önerilen):** 4c'yi çalıştırdıktan sonra Colab menüsünden
**Dosya → GitHub'da bir kopya kaydet** → aynı depo + aynı dal
(`claude/qnn-model-conversion-snapdragon7-rsk8og`). Colab hücre çıktılarını da
kaydeder; Claude not defterini depodan okuyup dökümü görür. Ek bir şey gerekmez.

**Yol 2:** Aşağıdaki hücre `sdk_help.txt`'yi telefona indirir; dosyayı sohbete
ek olarak gönderin.

In [ ]:
#@title 4d) sdk_help.txt dosyasını indir
from google.colab import files
files.download('/content/sdk_help.txt')

## 5) Modeli indir (civitai / HF / düz link)

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ("CIVITAI_TOKEN", "HF_TOKEN"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception:
            pass
except Exception:
    pass

!python scripts/fetch_model.py --url "$SAFETENSORS_URL" --output work/input.safetensors

## 6) Dönüştür (uçtan uca)

**Uyarı:** En uzun adım — UNet + VAE kuantizasyonu ~25-30 dk sürebilir.
Sekmeyi açık tutun. (`REBUILD_BIN` işaretliyse sadece .bin'ler ~3 dk'da yenilenir.)

In [ ]:
import os
os.environ["DSP_ARCH"] = DSP_ARCH
os.environ["REBUILD_BIN"] = "1" if REBUILD_BIN else "0"
os.environ["UNET_MODE"] = UNET_MODE
os.environ["QUANT_EXTRA"] = QUANT_EXTRA
print("UNET_MODE =", UNET_MODE)
print("DSP_ARCH =", DSP_ARCH or "(tier varsayilani -> min=v68)",
      "| REBUILD_BIN =", REBUILD_BIN)

!chmod +x convert_all.sh scripts/*.sh
!./convert_all.sh work/input.safetensors "$MODEL_NAME" "$TIER" "$RESOLUTIONS"

import glob
print("Üretilen:", glob.glob("dist/*.zip"))


## 7) Hugging Face'e yükle

`HF_REPO` boşsa, **model isminden otomatik repo** oluşturulur: `<kullanıcı_adın>/MODEL_NAME` — ve model oraya yüklenir (küçük bir model kartıyla). Belirli bir repoya yüklemek istersen `HF_REPO` alanını doldur. Gerekli: Colab Secrets'ta write izinli `HF_TOKEN`.

In [ ]:
import glob, os
zips = sorted(glob.glob("dist/*.zip"))
assert zips, "dist/ içinde zip yok — 6. adım başarısız olmuş olabilir."
zip_path = zips[-1]

# HF_TOKEN'i Secrets'tan al (yazma izinli olmalı)
try:
    from google.colab import userdata
    t = userdata.get("HF_TOKEN")
    if t:
        os.environ["HF_TOKEN"] = t
except Exception:
    pass

priv = "--private" if HF_PRIVATE else ""
if HF_REPO.strip():
    # Doğrudan verilen repoya yükle
    cmd = f'python scripts/upload_hf.py --repo "{HF_REPO}" --name "{MODEL_NAME}" --file "{zip_path}" {priv}'
elif UPLOAD_MODE.startswith("Koleksiyon"):
    # MOD B: tek koleksiyon reposu, her model alt klasörde
    cmd = f'python scripts/upload_hf.py --collection "{COLLECTION_REPO}" --name "{MODEL_NAME}" --file "{zip_path}" {priv}'
else:
    # MOD A: model adından ayrı repo
    cmd = f'python scripts/upload_hf.py --name "{MODEL_NAME}" --file "{zip_path}" {priv}'
print(">", cmd)
os.system(cmd)
print("ZIP:", zip_path)

## 8) (İsteğe bağlı) ZIP'i doğrudan bilgisayara indir

In [ ]:
from google.colab import files
import glob
zips = sorted(glob.glob("dist/*.zip"))
if zips:
    files.download(zips[-1])